# FinDirector — Directive Model Evaluation on Colab

Evaluate the fine-tuned Qwen 2.5 7B + LoRA directive model on the 156-example
held-out test set (greedy decoding, temperature 0).

**Setup:**
- Runtime: L4 GPU (24 GB VRAM)
- Estimated time: 5-20 minutes
- Cost: ~$0.15 in Colab compute units

**Prerequisites:**
- Colab Pro subscription with L4 GPU access
- Secrets configured: `HF_TOKEN` (adapter + dataset repos are private)

**Pipeline this notebook runs:**
1. Verify GPU allocation
2. Clone the FinDirector repo
3. Install requirements
4. Load `HF_TOKEN` from Colab Secrets manager
5. Run `scripts/evaluate_directive_model.py` (saves results/ files)
6. Inspect any failures (misclassifications + parse errors)

## Step 1 — Verify GPU

In [ ]:
!nvidia-smi

## Step 2 — Clone the FinDirector Repo

Get the latest evaluation script and config from GitHub.

In [ ]:
# Clone the repo (public HTTPS clone, no auth needed for public repos)
!git clone https://github.com/ShantanuBapat/findirector.git
%cd findirector
!git log -1 --oneline  # Show the latest commit for reference

## Step 3 — Install Requirements

Install all dependencies. Colab has some pre-installed but we force upgrade
to match our pinned versions (this also picks up scikit-learn for metrics).

**Note:** Colab ships an old `torchao` that PEFT rejects on adapter load. Upgrade
it here, then **Runtime -> Restart session** before running the eval so the fresh
version is the one imported.

In [ ]:
# Install all requirements
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
# Fix PEFT's torchao version check (Colab ships < 0.16.0). Restart runtime after.
!pip install -q --upgrade torchao

## Step 4 — Load Secrets from Colab Secrets Manager

Access `HF_TOKEN` without hardcoding it. Needed because both the adapter
(`AlHindi/findirector-directive-lora`) and the dataset
(`AlHindi/findirector-splits`) are private repos.

In [ ]:
import os
from google.colab import userdata

# Load secret into environment variable
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Verify (without printing the actual value)
assert os.environ["HF_TOKEN"].startswith("hf_"), "HF_TOKEN not loaded correctly"

print("Secret loaded successfully")
print(f"HF_TOKEN length: {len(os.environ['HF_TOKEN'])}")

## Step 5 — Run Evaluation

Execute the evaluation script. It downloads the LoRA adapter + test split from
HF Hub, generates on all 156 examples, prints the report, and saves:
- `results/eval_<date>.json` (metrics)
- `results/eval_<date>_details.jsonl` (per-example, with raw output)

**Expected duration:** 5-20 minutes on L4 GPU.

In [ ]:
# Run evaluation; report streams here, result files written to results/
!python -m scripts.evaluate_directive_model

## Step 6 — Inspect Failures

Load the per-example detail file and print every failure (misclassification or
parse error), including the raw generated output. Set `DETAILS` to the dated
file the run just wrote.

In [ ]:
import glob
import json

# Pick the most recent details file the eval run wrote
detail_files = sorted(glob.glob("results/eval_*_details.jsonl"))
DETAILS = detail_files[-1]
print(f"Reading {DETAILS}\n" + "=" * 70)

with open(DETAILS) as f:
    rows = [json.loads(line) for line in f if line.strip()]

failures = [r for r in rows if not r["correct"]]
print(f"{len(failures)} failures out of {len(rows)}")

for i, r in enumerate(failures, 1):
    # query stores the full prompt; strip to the user's actual question
    q = r["query"].split("Query:", 1)[-1].strip()
    verdict = r["predicted"] if r["status"] == "ok" else f"PARSE_ERROR({r['status']})"
    print(f"\n[{i}] true={r['truth']}  ->  got={verdict}")
    print(f"    query: {q}")
    print(f"    raw:   {r['raw_output'][:300]}")
    print("-" * 70)

## Done!

The report and the failure listing above show the model's classification quality
and the specific queries it got wrong.

**Result files written to `results/`:**
- `eval_<date>.json` — metrics
- `eval_<date>_details.jsonl` — per-example detail (download to commit)

**Next steps (Session 2.6 analysis):**
- Read per-code accuracy and the confusion matrix
- Inspect failures (see the list above) — separate real errors from label noise
- Document findings and commit the results